# Prediction of Product Sales

- Author: Salam

## Preprocessing

**Important - Avoiding Data Leakage:** in Parts 2-4 we cleaned the data and filled missing values with simple placeholders *before* doing any train/test split. That approach is fine for exploratory analysis, but it is NOT appropriate for modeling, because:
- Any imputation statistic (like a median or mode) calculated on the FULL dataset would "leak" information from the test set into the training process.
- Best practice is to perform the train/test split FIRST, and then fit any imputers/scalers/encoders ONLY on the training data.

Therefore, in this notebook we will load a **fresh, unmodified copy** of the original dataset and restart our cleaning process correctly, with imputation occurring after the split.

In [ ]:
# Mount Google Drive (Colab specific - skip if running locally)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Imports
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer

from sklearn import set_config
set_config(transform_output='pandas')  # keep outputs as DataFrames

pd.set_option('display.max_columns', 100)

### Load a Fresh Version of the Original Dataset

In [ ]:
# Load a FRESH copy of the original, uncleaned dataset using pd.read_csv()
# Adjust this path to wherever you saved the CSV in your Drive
fpath = '/content/drive/MyDrive/AXSOSACADEMY/02-IntroML/Project1/Data/sales_predictions_2023.csv'

df = pd.read_csv(fpath)
df.head()

In [ ]:
# Confirm we are starting fresh - nulls should still be present
# and Item_Fat_Content should still have its inconsistent categories
print(df.isna().sum())
print()
print(df['Item_Fat_Content'].unique())

### Deal with Duplicates and Inconsistent Categorical Data (Before the Split)

Per the instructions, duplicates and inconsistent categorical labels should be handled BEFORE splitting the data (these are data quality issues, not statistics that could leak information - fixing a typo like `'low fat'` -> `'Low Fat'` does not use any information from the target or about train vs. test rows).

In [ ]:
# Check for and drop any duplicate rows
print(f"Duplicate rows found: {df.duplicated().sum()}")
df = df.drop_duplicates()

In [ ]:
# Fix inconsistent categories in Item_Fat_Content
fat_content_map = {
    'low fat': 'Low Fat',
    'LF': 'Low Fat',
    'reg': 'Regular'
}
df['Item_Fat_Content'] = df['Item_Fat_Content'].replace(fat_content_map)

# Confirm only 2 consistent categories remain
df['Item_Fat_Content'].value_counts()

### Identify Features (X) and Target (y)

In [ ]:
# Target: Item_Outlet_Sales
y = df['Item_Outlet_Sales']

# Features: drop the target AND Item_Identifier
# (Item_Identifier has very high cardinality - 1,559 unique product IDs -
#  with no real predictive meaning, as discussed in Part 4's Feature
#  Inspection. We drop it here to avoid creating 1,559 one-hot columns.)
X = df.drop(columns=['Item_Outlet_Sales', 'Item_Identifier'])

X.head()

### Perform the Train-Test Split

In [ ]:
# Split the data BEFORE any imputation, to avoid data leakage
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

print(f"Training: {X_train.shape}")
print(f"Testing:  {X_test.shape}")

### Confirm Missing Values (on Training Data Only)

We only inspect/fit on the training data from this point forward, to make sure no information from the test set influences our preprocessing decisions.

In [ ]:
# Check for missing values in the training features
X_train.isna().sum()

- `Item_Weight` and `Outlet_Size` have missing values, consistent with what we found during EDA in Parts 2 and 4. These will be handled by `SimpleImputer` inside our preprocessing pipeline below (imputation statistics like the median will be calculated using ONLY the training data).

## Create a Preprocessing Object

### Preprocessing Pipeline for Numerical Features

Our numeric pipeline will fill in missing values with the **median** (using `SimpleImputer`), then scale the features with `StandardScaler`.

In [ ]:
# Save list of numeric columns
num_cols = X_train.select_dtypes('number').columns
num_cols

In [ ]:
# Constructing numeric preprocessing objects
num_imputer = SimpleImputer(strategy='median')
scaler = StandardScaler()

num_pipe = make_pipeline(num_imputer, scaler)
num_tuple = ('num', num_pipe, num_cols)
num_tuple

### Preprocessing Pipeline for Categorical Features

Our categorical pipeline will fill in missing values with the placeholder `'MISSING'` (using `SimpleImputer`), then one-hot encode the categorical features.

In [ ]:
# Save list of categorical columns
cat_cols = X_train.select_dtypes('object').columns
cat_cols

In [ ]:
# Constructing categorical preprocessing objects
cat_imputer = SimpleImputer(strategy='constant', fill_value='MISSING')
cat_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

cat_pipe = make_pipeline(cat_imputer, cat_encoder)
cat_tuple = ('cat', cat_pipe, cat_cols)
cat_tuple

### Combine into a Column Transformer

In [ ]:
# Define a column transformer that applies the appropriate pipeline
# to numeric vs. categorical columns
preprocessor = ColumnTransformer(
    [num_tuple, cat_tuple],
    verbose_feature_names_out=False
)
preprocessor

### Fit the Preprocessor on Training Data and Transform Both Sets

We `fit_transform` on the **training data only**, then use the already-fitted preprocessor to `transform` the **test data**. This ensures all imputation statistics (medians) and encoding categories are learned exclusively from the training set.

In [ ]:
# Fit on training data and transform it
X_train_processed = preprocessor.fit_transform(X_train)

# Transform the test data using the already-fitted preprocessor
X_test_processed = preprocessor.transform(X_test)

print(f"Processed training shape: {X_train_processed.shape}")
print(f"Processed test shape:     {X_test_processed.shape}")

X_train_processed.head()

## Summary

In this notebook, we:
1. Reloaded the **original, fresh** dataset to avoid any data leakage from our earlier exploratory cleaning.
2. Dropped duplicates and fixed inconsistent `Item_Fat_Content` categories (safe to do before the split, since these are simple data-quality fixes).
3. Defined `X` (features, dropping `Item_Identifier` due to high cardinality) and `y` (target = `Item_Outlet_Sales`).
4. Performed the **train-test split FIRST**, before any imputation.
5. Built a `ColumnTransformer` preprocessing object:
   - Numeric features: median imputation -> `StandardScaler`
   - Categorical features: `'MISSING'` placeholder imputation -> `OneHotEncoder`
6. Fit the preprocessor on the training data only, then used it to transform both the training and test sets.

We will finalize this project in Part 6, where we use `X_train_processed`, `X_test_processed`, `y_train`, and `y_test` (or a full model pipeline including the `preprocessor`) to build and evaluate Linear Regression and Random Forest models.